# SQLAlchemy: Alembic
## Intro

We vertrekken vanuit hetzelfde voorbeeld uit de eerste SQLAlchemy les. We gebruiken SQLAlchemy ORM om op een declaratieve manier een `user_account` table te definiëren met een aantal eenvoudige kolommen. We voeren ook alvast wat voorbeeld data toe.

Stel je voor dat deze tabel in gebruik is door een applicatie.

In [12]:
from sqlalchemy import String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

engine = create_engine("sqlite:///sqlite.db", echo=True)

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[str]

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

2025-10-30 07:40:22,583 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-30 07:40:22,583 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2025-10-30 07:40:22,583 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-30 07:40:22,584 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2025-10-30 07:40:22,584 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-30 07:40:22,585 INFO sqlalchemy.engine.Engine COMMIT
2025-10-30 07:40:22,585 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-30 07:40:22,585 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2025-10-30 07:40:22,586 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-30 07:40:22,586 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2025-10-30 07:40:22,586 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-30 07:40:22,587 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30) NOT NULL, 
	fullname VARCHAR NOT 

In [13]:
from sqlalchemy import insert
from sqlalchemy.orm import Session

with Session(engine) as session:
    session.execute(
        insert(User),
        [
            {"name": "wwhite", "fullname": "Walter White"},
            {"name": "jpinkman", "fullname": "Jesse Pinkman"},
            {"name": "gfring", "fullname": "Gus Fring"},
            {"name": "hschrade", "fullname": "Hank Schrader"},
            {"name": "mehrmant", "fullname": "Mike Ehrmantraut"},
        ],
    )
    session.commit()

2025-10-30 07:40:26,847 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-30 07:40:26,848 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2025-10-30 07:40:26,849 INFO sqlalchemy.engine.Engine [generated in 0.00049s] [('wwhite', 'Walter White'), ('jpinkman', 'Jesse Pinkman'), ('gfring', 'Gus Fring'), ('hschrade', 'Hank Schrader'), ('mehrmant', 'Mike Ehrmantraut')]
2025-10-30 07:40:26,849 INFO sqlalchemy.engine.Engine COMMIT


Alles werkt perfect maar nu krijg je als ontwikkelaar een nieuwe feature requirement...

Voor elke `User` moet nu ook de leeftijd worden opgeslagen.

Geen probleem, we breiden de `User` class gewoon uit en beginnen te werken aan de bijhorende logica in de applicatie die de leeftijd zal invoeren en gebruiken.

In [15]:
class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[str]
    age: Mapped[int]

Base.metadata.create_all(engine)

2025-10-30 07:43:36,333 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-30 07:43:36,334 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2025-10-30 07:43:36,335 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-30 07:43:36,335 INFO sqlalchemy.engine.Engine COMMIT


Helaas is het niet zo eenvoudig. De `user_account` table bestaat al dus SQLAlchemy doet helemaal niets tijdens `create_all`.

We kunnen ook niet zo maar de `age` kolom toevoegen want deze kolom is `NOT NULL` maar alle bestaande rijen zullen initieel `NULL zijn!

Precies om dit probleem op te lossen is Alembic gemaakt.

Alembic is een migratie-tool voor SQLAlchemy die helpt om wijzigingen in je database-schema (zoals nieuwe tabellen, kolommen of constraints) op een gecontroleerde en herhaalbare manier door te voeren. In plaats van manueel SQL-scripts te schrijven, gebruik je Alembic om migraties te genereren, beheren en toepassen via versiebeheer. Zo blijft je database consistent met de evolutie van je Python-modellen in SQLAlchemy.

## Voorbereiding
### Alembic

Alembic is een [apart Python package](https://pypi.org/project/alembic/) en installeer je, bvb. met `uv`:

```
uv add alembic
```

### Migration Environment

Een **migration environment** in Alembic is de structuur en configuratie die Alembic nodig heeft om migraties uit te voeren binnen jouw project.

De basis-structuur van deze migration environment maak je eenmalig aan met het commando `alembic init`.

Je project krijgt dan de bvb. de volgende structuur:

```
yourproject/
    alembic.ini
    pyproject.toml
    alembic/
        env.py
        README
        script.py.mako
        versions/
            3512b954651e_add_account.py
            2b1ae634e5cd_add_order_id.py
            3adcc9a56557_rename_username_field.py
```

- `alembic.ini` – Het hoofdconfiguratiebestand.
- `alembic/env.py` – Het script dat wordt uitgevoerd voor elke migratie.
- `alembic/script.py.mako` – Een template voor nieuwe migratiebestanden. [Mako](https://www.makotemplates.org/) is een templating library (net als Jinja2!).
- `alembic/versions/` – Zal al de gegenereerde migratiebestanden bevatten.

Kort gezegd: het migratie environment is het **technische kader** waarin Alembic migraties kan aanmaken, bijhouden en toepassen — vergelijkbaar met een “projectconfiguratie” voor databaseversiebeheer.

```bash
alembic init alembic

cat alembic.ini
ls -lR alembic
```

```bash
vi alembic.ini

sqlalchemy.url = postgresql://postgres@localhost/syntra
```

# Eerste Migratie - column toevoegen

```bash
alembic revision -m "Add user_account.age column"
```

```python
def upgrade() -> None:
    """Upgrade schema."""
    op.add_column("user_account", sa.Column("age", sa.Integer))


def downgrade() -> None:
    """Downgrade schema."""
    op.drop_column("user_account", "age")
```

## Upgrade

```
alembic upgrade head

psql syntra

\dt
\d user_account

select * from user_account;
select * from alembic_version;
```

## Downgrade

```bash
alembic downgrade -1

alembic upgrade +1
```

In [ ]:
with Session(engine) as session:
    session.add(User(name="sgoodman", fullname="Saul Goodman", age=42))
    session.commit()

In [ ]:
from sqlalchemy import update

with Session(engine) as session:
    session.execute(
        update(User),
        [
            {"id": 1, "age": 50},
            {"id": 2, "age": 26},
            {"id": 3, "age": 52},
            {"id": 4, "age": 48},
            {"id": 5, "age": 55},
        ],
    )
    session.commit()

# Tweede Migratie - column aanpassen

- `age` blijkt niet erg handig want moet elk jaar aangepast worden
- Nieuwe requirement: vervangen door geboortejaar, `year_of_birth`

In [ ]:
from datetime import datetime

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[str]
    year_of_birth: Mapped[int]

```bash
alembic revision -m "Migrate user_account.age to year_of_birth"
```

```python
def upgrade() -> None:
    """Upgrade schema."""
    op.add_column("user_account", sa.Column("year_of_birth", sa.Integer))
    op.drop_column("user_account", "age")

def downgrade() -> None:
    """Downgrade schema."""
    op.add_column("user_account", sa.Column("age", sa.Integer))
    op.drop_column("user_account", "year_of_birth")
```

In [ ]:
Maar ... we willen de data van `age` _migreren_ naar `year_of_birth` (en terug bij downgrade)

```python
def upgrade() -> None:
    """Upgrade schema."""
    op.add_column(
        "user_account", sa.Column("year_of_birth", sa.Integer(), nullable=True)
    )
    current_year = datetime.date.today().year
    op.execute(
        sa.text(
            "UPDATE user_account SET year_of_birth = :current_year - age"
        ).bindparams(current_year=current_year)
    )
    op.alter_column(
        "user_account", "year_of_birth", existing_type=sa.Integer(), nullable=False
    )
    op.drop_column("user_account", "age")


def downgrade() -> None:
    """Downgrade schema."""
    op.add_column("user_account", sa.Column("age", sa.Integer(), nullable=True))
    current_year = datetime.date.today().year
    op.execute(
        sa.text(
            "UPDATE user_account SET age = :current_year - year_of_birth"
        ).bindparams(current_year=current_year)
    )
    op.alter_column("user_account", "age", existing_type=sa.Integer(), nullable=False)
    op.drop_column("user_account", "year_of_birth")

# Andere alembic commandos

```bash
$ alembic current
INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Will assume transactional DDL.
65d0f854b8f7 (head)
```

```bash
$ alembic history
f6ed66efa7c4 -> 65d0f854b8f7 (head), Migrate user_account.age to year_of_birth
<base> -> f6ed66efa7c4, Add user_account.age column
```

```bash
$ alembic current --verbose
INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Will assume transactional DDL.
Current revision(s) for postgresql://postgres@localhost/syntra:
Rev: 65d0f854b8f7 (head)
Parent: f6ed66efa7c4
Path: /Users/dfabrice/dev/github/syntra/2_sqlalchemy/alembic/versions/65d0f854b8f7_migrate_user_account_age_to_year_of_.py

    Migrate user_account.age to year_of_birth
    
    Revision ID: 65d0f854b8f7
    Revises: f6ed66efa7c4
    Create Date: 2025-08-12 08:26:02.359420
```

# Autogenerate

Model (mapped classes) vergelijken met huidig schema in de database.

```python
# models/syntra.py

from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
    pass


class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[str]
    year_of_birth: Mapped[int]

    addresses: Mapped[list["Address"]] = relationship(back_populates="user")


class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("user_account.id"))
    email_address: Mapped[str]

    user: Mapped[User] = relationship(back_populates="addresses")
```

```python
# alembic/env.py

from models.syntra import Base
target_metadata = Base.metadata
```

```bash
$ alembic revision --autogenerate -m "Add address table"
INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Will assume transactional DDL.
INFO  [alembic.autogenerate.compare] Detected added table 'address'
INFO  [alembic.ddl.postgresql] Detected sequence named 'user_account_id_seq' as owned by integer column 'user_account(id)', assuming SERIAL and omitting
  Generating
  alembic/versions/80ec19433167_add_address_table.py ...  done
```

## Wat kan Autogenerate? 

### Wel
- Table en Column toevoegen/verwijderen
- (Non-)Nullable Column aanpassingen
- Aanpassingen aan Indexes, Unique Constraints en Foreign Key Constraints
- Column Type aanpassingen

### Niet
- Table name aanpassing. Wordt een drop+create (!!)
- Column name aanpassing. Idem
- Anonieme Constraints
- Speciale SQLAlchemy types, bvb `Enum`
- Sequence toevoegen/verwijderen

### Voorbeeld

```python
# models/syntra.py

class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    userid: Mapped[str] = mapped_column(String(30))      # <------
    fullname: Mapped[str]
    year_of_birth: Mapped[int]

    addresses: Mapped[list["Address"]] = relationship(back_populates="user")
```

```bash
$ alembic revision --autogenerate -m "Test"
INFO  [alembic.autogenerate.compare] Detected added column 'user_account.userid'
INFO  [alembic.autogenerate.compare] Detected removed column 'user_account.name'
```

```python
# alembic/versions/4b839c3abc99_test.py

def upgrade() -> None:
    """Upgrade schema."""
    op.add_column('user_account', sa.Column('userid', sa.String(length=30), nullable=False))
    op.drop_column('user_account', 'name')
```

Wat we in de plaats willen:

```python
def upgrade() -> None:
    """Upgrade schema."""
    op.alter_column("user_account", "name", new_column_name="userid")


def downgrade() -> None:
    """Downgrade schema."""
    op.alter_column("user_account", "userid", new_column_name="name")
```

# In the praktijk

## `sqlalchemy.url` beveiligen

- `alembic.ini` gaat in version control
- productie en test omgevingen zullen een passwoord hebben ...

URL dynamisch instellen vanuit `env.py`, bvb. met een environment variable:

```python
# alembic/env.py

if db_url := os.getenv("DB_URL"):
    config.set_main_option("sqlalchemy.url", db_url)
```

## Migratie detecteren

Is een migratie nodig op basis van de huidige code changes? (Zonder `alembic revision` te gebruiken)

```bash
$ alembic check
INFO  [alembic.autogenerate.compare] Detected added column 'user_account.userid'
INFO  [alembic.autogenerate.compare] Detected removed column 'user_account.name'
ERROR [alembic.util.messaging] New upgrade operations detected: [('add_column', None, 'user_account', Column('userid', String(length=30), table=<user_account>, nullable=False)), ('remove_column', None, 'user_account', Column('name', VARCHAR(length=30), table=<user_account>, nullable=False))]
  FAILED: New upgrade operations detected: [('add_column', None, 'user_account',
  Column('userid', String(length=30), table=<user_account>, nullable=False)),
  ('remove_column', None, 'user_account', Column('name', VARCHAR(length=30),
  table=<user_account>, nullable=False))]
```

```bash
$ alembic check
No new upgrade operations detected.
```

## Offline mode

Bvb. wanneer database aanpassingen gereviewed en/of uitgevoerd moeten worden door een 'database team'.

```bash
$ alembic history
88535212fcd9 -> 80ec19433167 (head), Add address table
65d0f854b8f7 -> 88535212fcd9, Add address table
f6ed66efa7c4 -> 65d0f854b8f7, Migrate user_account.age to year_of_birth
<base> -> f6ed66efa7c4, Add user_account.age column

$ alembic upgrade f6ed66efa7c4:65d0f854b8f7 --sql
INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Generating static SQL
INFO  [alembic.runtime.migration] Will assume transactional DDL.
BEGIN;

INFO  [alembic.runtime.migration] Running upgrade f6ed66efa7c4 -> 65d0f854b8f7, Migrate user_account.age to year_of_birth
-- Running upgrade f6ed66efa7c4 -> 65d0f854b8f7

ALTER TABLE user_account ADD COLUMN year_of_birth INTEGER;

UPDATE user_account SET year_of_birth = 2025 - age;

ALTER TABLE user_account ALTER COLUMN year_of_birth SET NOT NULL;

ALTER TABLE user_account DROP COLUMN age;

UPDATE alembic_version SET version_num='65d0f854b8f7' WHERE alembic_version.version_num = 'f6ed66efa7c4';

COMMIT;
```

## Naamgeving van Constraints

Naam wordt gekozen door de database.

Voorbeeld:

```python
class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("user_account.id")) # <-------
```

SQL (via metadata.create_all() of alembic):

```sql
CREATE TABLE address (
    id SERIAL NOT NULL, 
    user_id INTEGER NOT NULL, 
    email_address VARCHAR NOT NULL, 
    PRIMARY KEY (id), 
    FOREIGN KEY(user_id) REFERENCES user_account (id)
);
```

Foreign Key constraint:

```
syntra=# \d address;
                                     Table "public.address"
    Column     |       Type        | Collation | Nullable |               Default               
---------------+-------------------+-----------+----------+-------------------------------------
 id            | integer           |           | not null | nextval('address_id_seq'::regclass)
 user_id       | integer           |           | not null | 
 email_address | character varying |           | not null | 
Indexes:
    "address_pkey" PRIMARY KEY, btree (id)
Foreign-key constraints:
    "address_user_id_fkey" FOREIGN KEY (user_id) REFERENCES user_account(id)
```

Problemen:
- Juiste naam gebruiken bij aanpassingen (bvb `DROP CONSTRAINT` of `op.drop_constraint()`)
- Inconsistente namen bij verschillende databases

Oplossing: vaste naming convention in SQLAlchemy/Alembic

```python

class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("user_account.id"))
    email_address: Mapped[str] = mapped_column(unique=True)     # <-------------
```

```bash
$ alembic revision --autogenerate -m "Add email_address unique constraint"
INFO  [alembic.autogenerate.compare] Detected added unique constraint 'uq_address_email_address' on '('email_address',)'


```

```python
def upgrade() -> None:
    """Upgrade schema."""
    op.create_unique_constraint(op.f('uq_address_email_address'), 'address', ['email_address'])
```

```
syntra=# \d address

    "uq_address_email_address" UNIQUE CONSTRAINT, btree (email_address)
```

## Migratie script en Hooks

`alembic init` maakt een standaard migratie script aan, `env.py`. Dit wordt tijdens de `alembic` commandos.

Reeds gebruikt om `DB_URL` dynamisch in te lezen.

In `env.py` is `context.configure()` de centrale plek waar je instelt hoe een migratie uitgevoerd wordt.

Dit gebeurt in je env.py en kan via zogeheten hooks — dat zijn parameters waarmee je Alembic extra logica laat uitvoeren tijdens het migratieproces.

'Hooks':
- Parameters waarmee Alembic extra logica kan uitvoeren tijdens het migratieproces
- Zijn 'callback' functies die je meegeeft aan `context.configure()`

Veelgebruikte voorbeelden:

### `include_object`

Bvb. om bestaande tables niet te droppen.

```sql
CREATE TABLE foo (bar INT);
```

```bash
$ alembic check

  FAILED: New upgrade operations detected: [('remove_table', Table('foo', MetaData(),
  Column('bar', INTEGER(), table=<foo>), schema=None))]
```

Oplossing:

```python
# alembic/env.py

def include_object(object, name, type_, reflected, compare_to):
    if type_ == "table" and reflected and compare_to is None:
        return False
    else:
        return True

# ...
    
    context.configure(
        ...
        include_object=include_object,
    )
```

```bash
$ alembic check

No new upgrade operations detected.
```

### `process_revision_directives`

Aangeroepen wanneer je een nieuwe migratie maakt met `alembic revision --autogenerate`.

Bvb. om lege migratie bestaande te vermijden.

```python
# alembic/env.py

def process_revision_directives(context, revision, directives):
    script = directives[0]
    if script.upgrade_ops.is_empty():
        directives[:] = []

# ...
    
    context.configure(
        ...
        process_revision_directives=process_revision_directives,
    )
```

```bash
$ alembic revision --autogenerate -m "Test"

# Geen bestand aangemaakt
```

# Extra
## Links

- https://alembic.sqlalchemy.org/en/latest/